# 03 — Supplier Risk Model

## tl;dr

The baseline model identifies **4 high-risk suppliers** using an explainable weighted score. The model combines observed performance with geographic, financial, and sourcing-dependency exposure. Scores prioritize investigation; they are not predictions of supplier failure.

## Context & Methods

This notebook asks: **Which suppliers deserve management attention when performance, exposure, and dependency are considered together?**

### Key Assumptions

- The six weights represent a hypothetical management risk appetite.
- Component scores are normalized to a 0–100 scale.
- A score of 35 or more is high risk for this synthetic portfolio.
- The result is a prioritization model, not a causal or probabilistic forecast.

### 1. Load scores and inspect model weights

In [1]:
from pathlib import Path
import pandas as pd

from src.supply_chain_risk.analytics import RISK_WEIGHTS, apply_weight_scenario, build_analysis, load_data

ROOT = Path.cwd()
DATA_DIR = ROOT / "data" / "raw"
tables, supplier_scores, part_risk, kpis = build_analysis(DATA_DIR)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

pd.Series(RISK_WEIGHTS, name='weight').rename_axis('component').to_frame()

,weight
component,
delivery_risk,0.30
quality_risk,0.20
lead_time_variability_risk,0.15
geographic_risk,0.15
financial_risk,0.10
dependency_risk,0.10


### 2. Review the highest baseline supplier risks

In [2]:
risk_columns = ["supplier_name", "country", "risk_tier", "risk_score", "delivery_risk", "quality_risk", "lead_time_variability_risk", "geographic_risk", "financial_risk", "dependency_risk"]
supplier_scores[risk_columns].head(12).round(1)

,supplier_name,country,risk_tier,risk_score,delivery_risk,quality_risk,lead_time_variability_risk,geographic_risk,financial_risk,dependency_risk
0,Aero Systems 10,Poland,High,40.50,35.90,42.10,45.60,45,42,35.00
1,Aero Components 04,India,High,39.50,24.60,45.30,48.20,39,47,52.50
2,Vector Systems 20,India,High,39.00,36.80,29.80,62.70,42,30,32.50
3,Vector Systems 34,Japan,High,36.30,22.70,36.80,64.60,22,41,50.00
4,Orion Dynamics 14,Canada,Medium,34.60,27.80,38.80,65.20,17,43,18.60
5,Vector Precision 17,United Kingdom,Medium,34.20,12.80,34.20,53.30,27,38,76.70
6,Atlas Precision 07,Mexico,Medium,32.40,15.00,36.70,62.10,27,42,30.00
7,Nova Materials 32,Malaysia,Medium,32.30,15.60,38.00,54.90,48,31,15.00
8,Summit Systems 37,Poland,Medium,32.30,27.80,31.20,46.30,41,29,17.50
9,Orion Precision 05,Mexico,Medium,32.10,27.60,34.80,39.30,22,38,38.30


### 3. Reconcile the weighted score for the top supplier

In [3]:
top_supplier = supplier_scores.iloc[0]
reconciliation = pd.DataFrame([{"component": component, "component_score": top_supplier[component], "weight": weight, "weighted_points": top_supplier[component] * weight} for component, weight in RISK_WEIGHTS.items()])
reconciliation.loc[len(reconciliation)] = {"component": "Total", "component_score": pd.NA, "weight": reconciliation["weight"].sum(), "weighted_points": reconciliation["weighted_points"].sum()}
reconciliation.round(3)

,component,component_score,weight,weighted_points
0,delivery_risk,35.90,0.30,10.77
1,quality_risk,42.08,0.20,8.41
2,lead_time_variability_risk,45.57,0.15,6.83
3,geographic_risk,45.00,0.15,6.75
4,financial_risk,42.00,0.10,4.20
5,dependency_risk,35.00,0.10,3.50
6,Total,<NA>,1.00,40.47


### 4. Inspect the risk-tier distribution

In [4]:
supplier_scores.groupby("risk_tier", observed=True).agg(suppliers=("supplier_id", "count"), spend=("spend", "sum"), average_score=("risk_score", "mean")).reindex(["High", "Medium", "Low"]).round(1)

,suppliers,spend,average_score
risk_tier,,,
High,4,"10,506,221.30",38.80
Medium,17,"47,758,873.00",30.20
Low,19,"64,687,545.50",19.30


## Results

The model surfaces a small high-risk group while retaining the component evidence needed to challenge each score. Reconciliation confirms that the headline score equals the sum of weighted component points, subject only to display rounding.

## Takeaways

- Explainability is more useful than a black-box ranking for supplier reviews.
- High scores indicate where to investigate, not proof that disruption will occur.
- Thresholds and weights should be owned and calibrated by the business.
- Sensitivity testing is required before treating the ordering as robust.